# ConstructDrawingAI - Google Colab Runner
This notebook allows you to run and test the complete pipeline of **ConstructDrawingAI** on a Google Colab instance, taking advantage of free cloud GPUs (e.g., T4) to speed up deep learning training and inference.

### Step 1: Select GPU Runtime
Go to **Runtime -> Change runtime type** and select **T4 GPU** (or any available GPU).

### Step 2: Clone Repository

In [10]:
!git clone https://github.com/morehosseini/ConstructDrawingAI.git
%cd ConstructDrawingAI

Cloning into 'ConstructDrawingAI'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 206 (delta 6), reused 204 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 2.47 MiB | 18.30 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/ConstructDrawingAI/ConstructDrawingAI


### Step 3: Install `uv` and Project Dependencies

In [11]:
# Install uv package manager
!pip install uv

# Sync all dependencies including deep learning, evaluation and ingest packages
!uv sync --extra dev --extra eval --extra perception --extra backend --extra ingest

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 217 packages in 1ms
Prepared 1 package in 535ms
Installed 132 packages in 457ms
 + accelerate==1.13.0
 + albucore==0.0.24
 + albumentations==2.0.8
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.13.0
 + ast-serialize==0.5.0
 + black==26.5.1
 + certifi==2026.5.20
 + cffi==2.0.0
 + cfgv==3.5.0
 + charset-normalizer==3.4.7
 + click==8.4.1
 + constructdrawing-ai==0.1.0 (from file:///content/ConstructDrawingAI/ConstructDrawingAI)
 + contourpy==1.3.3
 + coverage==7.14.1
 + cryptography==48.0.0
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.5.5
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + distlib==0.4.0
 + ezdxf==1.4.4
 + fastapi==0.136.3
 + filelock==3.29.0
 + fonttools==4.63.0
 + fsspec==2026.4.0
 + h11==0.16.0
 + hf-xet==1.5.0
 + httpcore==1.0.9
 + httptools==0.8.0
 + httpx==0.28.1
 + huggingface-hub==1.17.0
 + hydra-core==1.3.2
 + identify==

### Step 4: Run the Unit Tests
Verify the installation by running pytest.

In [12]:
!uv run pytest -q

........................................................................ [ 32%]
........................................................................ [ 65%]
........................................................................ [ 98%]
...                                                                      [100%]
=============================== warnings summary ===============================
.venv/lib/python3.12/site-packages/fastapi/testclient.py:1
  /content/ConstructDrawingAI/ConstructDrawingAI/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

tests/test_graphml_cir.py::test_graphml_to_cir_nodes_edges_roles_and_norm
  /content/ConstructDrawingAI/ConstructDrawingAI/datasets/preparers/graphml_cir.py:115: ResourceWarning: unclosed file <_io.BufferedReader name='/tmp/pytest-of-root/pytest-1/

### Step 5: Generate Synthetic Drawings
Before training, we generate a synthetic pilot dataset of 100 electrical drawing sheets (clean and degraded variants) with ground-truth labels.

In [13]:
!uv run python -m synthetic.generate --type electrical --n 100 --out datasets/synthetic/electrical_pilot

Generated 100 electrical sample(s) -> datasets/synthetic/electrical_pilot
  validator: 100/100 exact (PASS)
  devices/sample: min=6 max=56; circuits/sample: min=2 max=5
  every record stamped license='synthetic-owned' lane='commercial'


### Step 6: Export the Synthetic Dataset to YOLO Format

In [14]:
!MPLBACKEND=agg uv run python -m perception export --profile local_debug

INFO perception.dataset: exported YOLO dataset: 78 train / 25 val images, 1300 boxes, 46/14 samples -> /content/ConstructDrawingAI/ConstructDrawingAI/outputs/perception/detector/local_debug/yolo
exported YOLO dataset -> /content/ConstructDrawingAI/ConstructDrawingAI/outputs/perception/detector/local_debug/yolo
  78 train / 25 val tiles, 1300 boxes; 46/14 samples (train/val)


### Step 7: Train Model 1 (Symbol Detector)
Train a YOLOv11 detector on the synthetic drawings using the T4 GPU.

In [15]:
!MPLBACKEND=agg uv run python -m perception train-detector --profile local_debug

INFO perception.dataset: exported YOLO dataset: 78 train / 25 val images, 1300 boxes, 46/14 samples -> /content/ConstructDrawingAI/ConstructDrawingAI/outputs/perception/detector/local_debug/yolo
INFO perception.detector: dataset ready: 78 train / 25 val tiles
WARNING eval.tracking: wandb not installed; ExperimentTracker is running in no-op mode.
INFO perception.detector: training detector: {'epochs': 40, 'imgsz': 1280, 'batch': 8, 'device': 0, 'name': 'local_debug'}
New https://pypi.org/project/ultralytics/8.4.96 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.12.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ConstructDrawingAI/ConstructDrawingAI/out

### Step 8: Evaluate the Symbol Detector

In [16]:
!MPLBACKEND=agg uv run python -m perception eval-detector --profile local_debug

SYNTHETIC-VALIDATION SCOREBOARD  —  PIPELINE SMOKE TEST (NOT real-drawing accuracy)
A number here means the training loop + L0->L1 handoff + CIR wiring work on
held-out SYNTHETIC data. It is NOT evidence the model reads real drawings.
The published-SOTA / published-frontier rows below were measured on REAL drawings
and are therefore NOT directly comparable to our synthetic row — they are shown for
context only. The meaningful comparison to SOTA is the REAL-DRAWING scoreboard.

(detection; held-out synthetic samples: 28; conditions: clean, degraded; model: cdai-detector-local_debug)

# Matrix evaluation leaderboard

_Legend: kind=measured -> computed by us on synthetic ground truth; kind=reported -> literature-cited (see source). No external APIs are called._

### mep - counting_exact_match  (higher is better)

| rank | model | condition | score | seeds | kind | source |
|---|---|---|---|---|---|---|
| 1 | oracle  <- best | clean | 100.0% | 1 | measured | measured by us (synthetic GT) |

### Step 9: Train Model 2 (Connectivity Graph)
Train the network to extract connectivity edges between electrical devices.

In [17]:
!MPLBACKEND=agg uv run python -m perception train-connectivity --profile local_debug

INFO perception.connectivity: connectivity training pairs: 5196 (train), 1756 (val)
WARNING eval.tracking: wandb not installed; ExperimentTracker is running in no-op mode.
INFO eval.tracking: [no-op tracker] step=0 metrics={'loss': 1.3993590816889845, 'val_edge_macro_f1': 0.013430330162283156, 'val_any_edge_f1': 0.20961145194274028}
INFO eval.tracking: [no-op tracker] step=1 metrics={'loss': 1.3892303318863561, 'val_edge_macro_f1': 0.013644115974985787, 'val_any_edge_f1': 0.21058091286307054}
INFO eval.tracking: [no-op tracker] step=2 metrics={'loss': 1.379718968737942, 'val_edge_macro_f1': 0.013872832369942197, 'val_any_edge_f1': 0.2095839915745129}
INFO eval.tracking: [no-op tracker] step=3 metrics={'loss': 1.37058092641133, 'val_edge_macro_f1': 0.01437908496732026, 'val_any_edge_f1': 0.19894055326662743}
INFO eval.tracking: [no-op tracker] step=4 metrics={'loss': 1.3616292141693385, 'val_edge_macro_f1': 0.015259895088221268, 'val_any_edge_f1': 0.20293554562858968}
INFO eval.tracking

### Step 10: Evaluate the Connectivity Graph

In [18]:
!MPLBACKEND=agg uv run python -m perception eval-connectivity --profile local_debug

SYNTHETIC-VALIDATION SCOREBOARD  —  PIPELINE SMOKE TEST (NOT real-drawing accuracy)
A number here means the training loop + L0->L1 handoff + CIR wiring work on
held-out SYNTHETIC data. It is NOT evidence the model reads real drawings.
The published-SOTA / published-frontier rows below were measured on REAL drawings
and are therefore NOT directly comparable to our synthetic row — they are shown for
context only. The meaningful comparison to SOTA is the REAL-DRAWING scoreboard.

(connectivity; held-out synthetic samples: 28; conditions: clean, degraded; model: cdai-connectivity-local_debug)

# Matrix evaluation leaderboard

_Legend: kind=measured -> computed by us on synthetic ground truth; kind=reported -> literature-cited (see source). No external APIs are called._

### mep - graph_edge_ap  (higher is better)

| rank | model | condition | score | seeds | kind | source |
|---|---|---|---|---|---|---|
| 1 | oracle  <- best | clean | 100.0% | 1 | measured | measured by us (synthetic GT) |